In [1]:
import os
import re
import logging

import isodate
from dotenv import load_dotenv
from googleapiclient.discovery import build

In [2]:
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)
logger = logging.getLogger(__name__)

In [3]:
def extract_video_id(url: str) -> str:
    """Extract and return the video ID from a YouTube URL."""
    patterns = [
        r"(?:v=)([A-Za-z0-9_-]{11})",
        r"youtu\.be/([A-Za-z0-9_-]{11})",
        r"shorts/([A-Za-z0-9_-]{11})",
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Could not extract a YouTube video ID from: {url}")

In [4]:
def get_video_metadata(url: str) -> dict:
    """Fetch and return metadata for a YouTube video via the Data API v3."""
    video_id = extract_video_id(url)
    logger.info("Fetching metadata for video_id=%s", video_id)

    youtube = build("youtube", "v3", developerKey=os.environ["YOUTUBE_API_KEY"])

    video_resp = youtube.videos().list(
        part="snippet,statistics,contentDetails",
        id=video_id,
    ).execute()

    if not video_resp.get("items"):
        raise ValueError(f"No video found for ID: {video_id}")

    item = video_resp["items"][0]
    snippet = item["snippet"]
    stats = item.get("statistics", {})
    content = item["contentDetails"]

    views = int(stats.get("viewCount", 0))
    likes = int(stats.get("likeCount", 0))
    comments = int(stats.get("commentCount", 0))

    duration_seconds = int(
        isodate.parse_duration(content["duration"]).total_seconds()
    )

    # Hashtags from description (MAX 20 for now)
    description = snippet.get("description", "")
    hashtags = re.findall(r"#\w+", description)[:20]

    channel_resp = youtube.channels().list(
        part="statistics",
        id=snippet["channelId"],
    ).execute()

    subscriber_count = 0
    if channel_resp.get("items"):
        channel_stats = channel_resp["items"][0].get("statistics", {})
        subscriber_count = int(channel_stats.get("subscriberCount", 0))

    engagement_rate = (
        round((likes + comments) / views * 100, 4) if views > 0 else 0.0
    )

    return {
        "video_id": video_id,
        "title": snippet["title"],
        "creator": snippet["channelTitle"],
        "upload_date": snippet["publishedAt"][:10],
        "duration_seconds": duration_seconds,
        "views": views,
        "likes": likes,
        "comments": comments,
        "hashtags": hashtags,
        "thumbnail_url": snippet.get("thumbnails", {}).get("high", {}).get("url", ""),
        "engagement_rate": engagement_rate,
        "subscriber_count": subscriber_count,
    }

In [5]:
TEST_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
metadata = get_video_metadata(TEST_URL)

2026-05-19 02:13:08,978 INFO Fetching metadata for video_id=dQw4w9WgXcQ
2026-05-19 02:13:08,981 INFO file_cache is only supported with oauth2client<4.0.0


In [6]:
for key, value in metadata.items():
    print(f"{key:<20} {value}")

video_id             dQw4w9WgXcQ
title                Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)
creator              Rick Astley
upload_date          2009-10-25
duration_seconds     214
views                1773727547
likes                19102692
comments             2438071
hashtags             ['#RickAstleyNever', '#RickAstley', '#NeverGonnaGiveYouUp', '#WheneverYouNeedSomebody', '#OfficialMusicVideo']
thumbnail_url        https://i.ytimg.com/vi/dQw4w9WgXcQ/hqdefault.jpg
engagement_rate      1.2144
subscriber_count     4500000


In [ ]:
REAL_URL = "https://www.youtube.com/watch?v=SVTPv4sI_Jc"
metadata = get_video_metadata(REAL_URL)

REAL_METADATA = {
    "video_id": metadata["video_id"],
    "title": metadata["title"],
    "creator": metadata["creator"],
    "engagement_rate": metadata["engagement_rate"],
}

print("REAL_METADATA =")
print(REAL_METADATA)

2026-05-19 02:17:10,734 INFO Fetching metadata for video_id=SVTPv4sI_Jc
2026-05-19 02:17:10,737 INFO file_cache is only supported with oauth2client<4.0.0


REAL_METADATA =
{'video_id': 'SVTPv4sI_Jc', 'title': "The CIA's Worst New Tech Idea?", 'creator': 'Veritasium', 'engagement_rate': 3.3747}


: 